[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_11_12/01_tag_11_12_zeitreihe_trainingssequenzen.ipynb)

# Tag 11 & 12 – 01: Eine Zeitreihe in Trainingssequenzen umwandeln

Zeitreihen liegen zunächst als **eine geordnete Folge** von Messwerten vor. Für überwachtes Lernen brauchen wir dagegen viele Beispiele der Form **Eingabe → Ziel**. Dieses Notebook erzeugt aus jeweils zehn vergangenen Werten den unmittelbar folgenden, elften Wert als Ziel.

## Lernziel und Datenform

Ein Fenster mit Länge $L=10$ wird zu einem Eingabebeispiel `X[i]`; der nächste Wert wird zum Ziel `y[i]`:

```text
Messwerte:  ...  x₀, x₁, ..., x₉, x₁₀, x₁₁, ...
Beispiel 0:  X[0] = [x₀, ..., x₉]      → y[0] = x₁₀
Beispiel 1:  X[1] = [x₁, ..., x₁₀]     → y[1] = x₁₁
```

Die Aufteilung erfolgt **chronologisch**: Der frühere Abschnitt wird zum Training, der spätere zum Testen. Ein zufälliges Mischen vor der Aufteilung wäre bei einer Zeitreihe problematisch, weil dann Informationen aus der Zukunft im Training landen könnten.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42

## Künstliche Messreihe erzeugen

Die Kurve kombiniert zwei Sinuswellen und leichtes Zufallsrauschen. Das steht beispielhaft für periodische Messwerte – etwa Energieverbrauch, Temperatur oder Sensordaten. Für dieselbe Zufallszahl entsteht immer dieselbe Reihe; dadurch bleiben die Effekte der Regler gut vergleichbar.

In [ ]:
def erzeuge_zeitreihe(anzahl_werte=240, rauschen=0.12, seed=RANDOM_STATE):
    """Erzeugt eine periodische, leicht verrauschte Zeitreihe."""
    rng = np.random.default_rng(seed)
    zeit = np.linspace(0, 8 * np.pi, anzahl_werte)
    werte = (
        np.sin(zeit)
        + 0.30 * np.sin(2.7 * zeit + 0.5)
        + rng.normal(loc=0.0, scale=rauschen, size=anzahl_werte)
    )
    return zeit, werte


zeit, werte = erzeuge_zeitreihe()
plt.figure(figsize=(12, 3.5))
plt.plot(werte, color='tab:blue', linewidth=1.6)
plt.title('Künstliche Zeitreihe')
plt.xlabel('Zeitindex')
plt.ylabel('Messwert')
plt.show()

## Chronologisch teilen und Fenster bauen

Zuerst teilen wir die **Rohwerte** zeitlich in Training und Test. Anschließend erzeugt die Hilfsfunktion aus jedem Abschnitt überlappende Fenster. Die Schleife beginnt bei `0` und endet bei `len(reihe) - fensterlaenge - 1`: Für jedes Eingabefenster muss noch genau ein nachfolgender Zielwert vorhanden sein.

Im Testabschnitt werden hier nur Testwerte verwendet. Das ist besonders anschaulich und strikt getrennt; in einer Produktions-Pipeline darf das erste Testfenster häufig auch die letzten bekannten Trainingswerte als Vorgeschichte enthalten.

In [ ]:
def erstelle_fenster(reihe, fensterlaenge):
    """Macht aus [x0, x1, ...] Paare aus Vergangenheit X und nächstem Wert y."""
    X, y = [], []
    for start in range(len(reihe) - fensterlaenge):
        X.append(reihe[start : start + fensterlaenge])
        y.append(reihe[start + fensterlaenge])
    return np.asarray(X), np.asarray(y)


FENSTERLAENGE = 10
TRAINING_ANTEIL = 0.70
trenngrenze = int(len(werte) * TRAINING_ANTEIL)
train_werte, test_werte = werte[:trenngrenze], werte[trenngrenze:]

X_train, y_train = erstelle_fenster(train_werte, FENSTERLAENGE)
X_test, y_test = erstelle_fenster(test_werte, FENSTERLAENGE)

print(f'Rohwerte: Training = {len(train_werte)}, Test = {len(test_werte)}')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')
print()
for i in range(3):
    print(f'Beispiel {i}: X_train[{i}] = {np.round(X_train[i], 2)}  →  y_train[{i}] = {y_train[i]:.2f}')

assert X_train.shape[1] == FENSTERLAENGE
assert len(X_train) == len(train_werte) - FENSTERLAENGE

## Interaktive Ansicht: Was wird ein Trainingsbeispiel?

Die blauen Punkte sind die Werte eines Eingabefensters `X`. Der orange Punkt ist der nächste Wert `y`, den ein späteres Modell vorhersagen soll. Mit den Reglern lässt sich beobachten, wie Rauschen, Fensterlänge und zeitliche Aufteilung die Anzahl und Form der Beispiele verändern.

In [ ]:
def zeige_fenster(fensterlaenge=10, training_anteil=70, rauschen=0.12, bereich='Training', beispiel_nummer=0):
    _, aktuelle_werte = erzeuge_zeitreihe(anzahl_werte=240, rauschen=rauschen)
    grenze = int(len(aktuelle_werte) * training_anteil / 100)
    train, test = aktuelle_werte[:grenze], aktuelle_werte[grenze:]
    X_train, y_train = erstelle_fenster(train, fensterlaenge)
    X_test, y_test = erstelle_fenster(test, fensterlaenge)

    X, y = (X_train, y_train) if bereich == 'Training' else (X_test, y_test)
    offset = 0 if bereich == 'Training' else grenze
    if len(X) == 0:
        print('Für diesen Bereich bleiben keine vollständigen Fenster. Wähle eine kürzere Fensterlänge.')
        return

    index = min(beispiel_nummer, len(X) - 1)
    x_positionen = np.arange(offset + index, offset + index + fensterlaenge)
    ziel_position = offset + index + fensterlaenge

    fig, (ax_gesamt, ax_beispiel) = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={'height_ratios': [2, 1]})
    ax_gesamt.plot(aktuelle_werte, color='0.55', linewidth=1.3, label='gesamte Zeitreihe')
    ax_gesamt.axvspan(0, grenze - 1, color='tab:blue', alpha=0.08, label='Training')
    ax_gesamt.axvspan(grenze, len(aktuelle_werte) - 1, color='tab:orange', alpha=0.08, label='Test')
    ax_gesamt.plot(x_positionen, X[index], 'o-', color='tab:blue', linewidth=2.5, label='Eingabe X')
    ax_gesamt.scatter(ziel_position, y[index], s=90, color='tab:orange', zorder=3, label='Ziel y')
    ax_gesamt.axvline(grenze - 0.5, color='black', linestyle='--', linewidth=1)
    ax_gesamt.set(title=f'{bereich}: Fenster {index} von {len(X) - 1}', xlabel='Zeitindex', ylabel='Messwert')
    ax_gesamt.legend(ncol=4, loc='upper right')

    ax_beispiel.plot(np.arange(1, fensterlaenge + 1), X[index], 'o-', color='tab:blue', label='X: vergangene Werte')
    ax_beispiel.scatter(fensterlaenge + 1, y[index], s=90, color='tab:orange', zorder=3, label='y: nächster Wert')
    ax_beispiel.axvline(fensterlaenge + 0.5, color='0.4', linestyle=':')
    ax_beispiel.set_xticks(np.arange(1, fensterlaenge + 2))
    ax_beispiel.set(title=f'Ein Beispiel: X hat die Form ({fensterlaenge},), y ist ein Skalar', xlabel='Position im Fenster', ylabel='Messwert')
    ax_beispiel.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

    print(f'Chronologische Trennung bei Index {grenze}: {grenze} Trainings- und {len(test)} Testwerte')
    print(f'X_{bereich.lower()}.shape = {X.shape}; y_{bereich.lower()}.shape = {y.shape}')
    if beispiel_nummer != index:
        print(f'Beispielnummer {beispiel_nummer} existiert nicht; angezeigt wird das letzte Fenster {index}.')
    print(f'X[{index}] = {np.round(X[index], 2)}')
    print(f'y[{index}] = {y[index]:.2f}')


widgets.interact(
    zeige_fenster,
    fensterlaenge=widgets.IntSlider(value=10, min=3, max=30, step=1, description='Fensterlänge'),
    training_anteil=widgets.IntSlider(value=70, min=55, max=85, step=5, description='Training (%)'),
    rauschen=widgets.FloatSlider(value=0.12, min=0.0, max=0.50, step=0.02, description='Rauschen'),
    bereich=widgets.ToggleButtons(options=['Training', 'Test'], description='Bereich'),
    beispiel_nummer=widgets.IntSlider(value=0, min=0, max=80, step=1, description='Fenster Nr.'),
);

## Kerngedanke

`X` ist nach der Umwandlung kein einzelner Zeitverlauf mehr, sondern eine Sammlung vieler überlappender Beispiele mit der Form `(anzahl_fenster, fensterlaenge)`. `y` enthält den jeweils passenden Folgewert mit der Form `(anzahl_fenster,)`. Genau dieses Paar kann anschließend beispielsweise an ein Dense-Netz, ein RNN, ein LSTM oder ein 1D-CNN übergeben werden.